# 04 Full trajectory from inputs only, with bounce and roll

Everything here starts from the 24 input columns of a shot (as in `test.csv`) and the trained final model saved by `make_submission.py` (`models/final_model.joblib`). No notebook cache is used.

**Flight** (`inrange.trajectory.shot_trajectory`). The model solves the shot's spin, spin-axis tilt and launch speed factor from its checkpoints and predicts apex and landing. The drawn path is the physics simulation with those fitted states, corrected in two ways so that it passes through what is known:

* **Time warp.** Piecewise linear and increasing, mapping the physics times at cp1 to cp4, apex and level landing onto the observed checkpoint times and the predicted `apex_t` and `landing_t`.
* **Position offset.** Per coordinate, a smooth monotone-slope (PCHIP) curve in physics time: 0 at launch, observed minus physics at the checkpoints, predicted minus physics at apex and landing, constant after landing. The height offset's slope at the apex is set so the drawn apex is a stationary point.

After level landing the ball keeps falling to the ground, which matters for the balcony tee T3 (4.125 m up). It then **bounces and rolls** following Penner's (2002b) run model.

**Scene frame for the renderers:** downrange from the common tee line, lateral from the middle of the four tees, height above ground.

In [1]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from inrange.io import INPUT_COLS, REPO_ROOT, load_test, load_train
from inrange.frame import add_shot_frame
from inrange.features import ball_speed
from inrange.render import (
    GALLERY_CRITERIA, ScenePath, nearest_tee, plotly_figure, select_gallery, write_gif, write_html,
)
from inrange.trajectory import PENNER, BounceParams, load_model, shot_trajectories, with_bounce

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220, "display.max_columns", 30, "display.precision", 3)
FIG_DIR = REPO_ROOT / "outputs" / "figures"
ANIM_DIR = REPO_ROOT / "outputs" / "animations"
ANIM_DIR.mkdir(parents=True, exist_ok=True)

model = load_model()
train, test = load_train(), load_test()
rows = pd.concat([train[INPUT_COLS].assign(split="train"), test[INPUT_COLS].assign(split="test")], ignore_index=True)
print(f"model choice: {model.choice}; {len(rows)} shots")

model choice: {'landing_pos': 'b', 'apex_pos': 'b', 'apex_t': 'b', 'landing_t': 'b', 'spin': 'a'}; 1050 shots


## 1. Stitching every shot

In [2]:
started = time.perf_counter()
trajectories = shot_trajectories(rows[INPUT_COLS], model)
print(f"{len(trajectories)} trajectories in {time.perf_counter() - started:.1f} s")

frame = add_shot_frame(rows[INPUT_COLS])
records = []
for i, tj in enumerate(trajectories):
    path, knots = tj.path, tj.knots.set_index("knot")
    record = {"split": rows.at[i, "split"], "tee_z": tj.tee_z}
    hit = 0.0
    for name in ["cp1", "cp2", "cp3", "cp4", "apex", "landing"]:
        point = path.loc[(path["t"] - knots.at[name, "t"]).abs().idxmin()]
        if name.startswith("cp"):
            target = [frame.at[i, f"{name}_{a}"] for a in "dlh"]
        else:
            target = [tj.events[name][a] for a in "dlh"]
        hit = max(hit, float(np.max(np.abs(point[["d", "l", "h"]].to_numpy(dtype=float) - target))),
                  abs(point["t"] - knots.at[name, "t"]) * 1000)
        for axis in "dlh":
            record[f"{name}_{axis}"] = knots.at[name, f"offset_{axis}"]
    airborne = path[(path["t"] > 0.05) & (path["t"] < tj.events["landing"]["t"])]
    record.update({
        "worst_hit_m_or_ms": hit,
        "apex_excess_m": path.loc[path["phase"].isin(["radar", "flight"]), "h"].max() - tj.events["apex"]["h"],
        "bounce_above_apex_m": path.loc[path["phase"] == "bounce", "h"].max() - tj.events["apex"]["h"]
        if (path["phase"] == "bounce").any() else -np.inf,
        "min_height_airborne_m": airborne["height"].min(),
        "warp_apex_s": knots.at["apex", "t"] - knots.at["apex", "tau"],
        "warp_landing_s": knots.at["landing", "t"] - knots.at["landing", "tau"],
        "apex_before_net": any("apex before the net" in n for n in tj.notes),
        "apex_knot_moved": any("apex knot moved" in n for n in tj.notes),
        "height_capped": any("height capped" in n for n in tj.notes),
        "cap_m": max([float(n.split("up to ")[1].split(" m")[0]) for n in tj.notes if "height capped" in n] or [0.0]),
        **tj.distances,
    })
    records.append(record)
stats = pd.DataFrame(records)

print("checks over all shots:")
print(f"  worst miss at checkpoints, apex and landing: {stats['worst_hit_m_or_ms'].max():.2e} (m or ms)")
print(f"  largest drawn flight height above the predicted apex: {stats['apex_excess_m'].max():.2e} m")
bounce_high = stats["bounce_above_apex_m"] > 0
print(f"  shots whose first bounce rises above their apex: {int(bounce_high.sum())} "
      f"(by up to {stats.loc[bounce_high, 'bounce_above_apex_m'].max():.2f} m)")
print(f"  lowest height above ground before level landing: {stats['min_height_airborne_m'].min():.2f} m")

offsets = stats[[f"{n}_{a}" for n in ["cp1", "cp2", "cp3", "cp4", "apex", "landing"] for a in "dlh"]].abs()
summary = pd.DataFrame({"median": offsets.median(), "95th pct": offsets.quantile(0.95), "max": offsets.max()})
print("\nposition corrections |predicted or observed minus physics| (m):")
print(summary.T.round(2).to_string())
print("\ntime warp at apex and landing (s):")
print(stats[["warp_apex_s", "warp_landing_s"]].abs().describe(percentiles=[0.5, 0.95]).T[["50%", "95%", "max"]].round(3))
print("\nspecial cases:", {k: int(stats[k].sum()) for k in ["apex_before_net", "apex_knot_moved", "height_capped"]})
print("height cap size where applied (m):", stats.loc[stats["height_capped"], "cap_m"].describe(percentiles=[0.5, 0.95]).round(3).to_dict())

1050 trajectories in 21.6 s


checks over all shots:
  worst miss at checkpoints, apex and landing: 7.11e-15 (m or ms)
  largest drawn flight height above the predicted apex: 0.00e+00 m
  shots whose first bounce rises above their apex: 1 (by up to 0.29 m)
  lowest height above ground before level landing: 0.05 m

position corrections |predicted or observed minus physics| (m):
          cp1_d  cp1_l  cp1_h  cp2_d  cp2_l  cp2_h  cp3_d  cp3_l  cp3_h  cp4_d  cp4_l  cp4_h  apex_d  apex_l  apex_h  landing_d  landing_l  landing_h
median      0.0   0.01   0.22    0.0   0.03   0.33    0.0   0.03   0.40    0.0   0.05   0.51    1.18    0.49    0.86       2.93       2.39        0.0
95th pct    0.0   0.23   0.72    0.0   0.29   1.05    0.0   0.23   1.40    0.0   0.32   1.91    5.90    2.72    4.01      11.54       9.16        0.0
max         0.0   0.58   1.26    0.0   0.82   2.27    0.0   1.62   3.68    0.0   2.54   6.89   13.72   10.01    8.70      20.30      31.20        0.0

time warp at apex and landing (s):
              

**Reading.** The drawn path meets every checkpoint and the submitted apex and landing to numerical precision, stays above ground in flight, and never rises above the submitted apex.

* **Checkpoint corrections** are small: the fitted physics already passes near the radar points, so these are decimetres at most for typical shots.
* **Apex and landing corrections** are the size of the difference between the final model and pure physics (metres), since the final model corrects physics with LightGBM.
* **Bounces above the apex:** one very low shot has a first bounce higher than its apex. Penner's turf-compliance rotation turns horizontal speed into vertical speed on shallow impacts, which is an extrapolation of a rule fitted to one steep impact.
* **Special cases:**
  * Low shots often have their apex before the net. The stitching handles this without change, but a few need the apex knot moved so that the time warp stays increasing.
  * Where the predicted apex sits just above a checkpoint that the ball passes while still climbing, the stitched height is capped at the apex, which leaves a short flat top.

## 2. Bounce and roll

There is **no bounce or roll data** in this competition, so this part is literature-based and cannot be validated against these shots. The model follows Penner (2002b), applied in the vertical plane of the ball's horizontal motion on flat ground:

| Quantity | Value | Source |
| --- | --- | --- |
| Coefficient of restitution for normal impact speed $v_n$ | $0.510 - 0.0375 v_n + 0.000903 v_n^2$ for $v_n \le 20$ m/s, else 0.120 | Penner (2002b, p. 933, eq. 5) |
| Turf compliance: impact frame rotated by | $\theta_c = 15.4° \,(v/18.6\,\text{m/s})(\phi/44.4°)$ | Penner (2002b, p. 934, eq. 8) |
| Friction coefficient | 0.40 (rolls out of the impact if above $\mu_c$) | Daish, as cited in Penner (2002b, p. 933) |
| Tangential rebound when rolling | $u' = \tfrac57 u - \tfrac27 r\omega$ | Penner (2002b, p. 933, eq. 3) |
| Bouncing ends below | 5 mm rebound height | Penner (2002b, p. 935) |
| Rolling deceleration | $\tfrac57 \rho_g g$, $\rho_g = 0.131$ | Penner (2002b, p. 935); greens span 0.065 to 0.196 (Penner, 2002a, p. 85) |

Penner's $\rho_g$ is a putting-green value, and he notes a fairway would be expected to have a larger one (2002b, p. 935), so rolls here are probably on the long side for a driving range. As a check of the implementation, the model reproduces Penner's first-bounce heights for two drives (1.17 m and 2.82 m; 2002b, p. 936), see `tests/test_trajectory.py`. For a wider range of turf behaviour, Biber et al. (2023, Table 3, p. 11) fitted restitution coefficients between 0.15 and 0.54 on teeing turf, depending on the model.

**References**

Biber, S. W., Jones, K. M., Champneys, A. R., Green, R., & Szalai, R. (2023). *Measurements and linearized models for golf ball bounce* (arXiv:2302.02758). arXiv. https://arxiv.org/abs/2302.02758

Penner, A. R. (2002a). The physics of putting. *Canadian Journal of Physics, 80*, 83–96. https://doi.org/10.1139/p01-137

Penner, A. R. (2002b). The run of a golf ball. *Canadian Journal of Physics, 80*(8), 931–940. https://doi.org/10.1139/p02-035

In [3]:
gallery_rows = test.reset_index(drop=True)
prediction = model.predict_full(gallery_rows)
gallery = select_gallery(gallery_rows, prediction.targets, prediction.states)
for slot, track in gallery.items():
    print(f"{slot:>13}: {track}  ({GALLERY_CRITERIA[slot]})")

by_id = {tj.track_id: tj for tj in trajectories}
table = []
for slot, track in gallery.items():
    tj = by_id[track]
    row = test[test["track_id"] == track].iloc[0]
    v = tj.impact_velocity
    table.append({
        "shot": slot, "tee": nearest_tee(row["launch_x"], row["launch_y"]),
        "ball speed (m/s)": float(ball_speed(row.to_frame().T.astype({c: float for c in INPUT_COLS if c != "track_id"})).iloc[0]),
        "spin (rpm)": tj.prediction["launch_spin_rate"],
        "carry (m)": tj.distances["carry"], "ground carry (m)": tj.distances["ground_carry"],
        "impact speed (m/s)": float(np.linalg.norm(v)),
        "impact angle from vertical (deg)": float(np.degrees(np.arctan2(np.hypot(v[0], v[1]), -v[2]))),
        "first bounce (m)": tj.distances["first_bounce_height"],
        "bounce (m)": tj.distances["bounce"], "roll (m)": tj.distances["roll"], "total (m)": tj.distances["total"],
    })
distances = pd.DataFrame(table).set_index("shot")
print(distances.round(1).to_string())

print("\nall 1050 shots, run after ground contact (m):")
print((stats["total"] - stats["ground_carry"]).describe(percentiles=[0.05, 0.5, 0.95]).round(1).to_dict())
print("hops per shot:", stats["n_bounces"].describe().round(1).to_dict())

        wedge: 71140f76-70ab-4870-b3c9-5f74b80272f6  (ball speed below 42 m/s and predicted spin at least 8500 rpm; the one with median predicted carry)
     mid iron: 0c52f92e-c904-4c4d-aa33-f28818a016ba  (ball speed 55 to 62 m/s, predicted spin 5000 to 7000 rpm, landing within 5 m of the launch line; median predicted carry)
       driver: 6df1d589-c1d7-4b59-ade9-4e1c9e967629  (ball speed at least 70 m/s; lowest predicted spin)
 strong curve: 7ffa783e-965c-4643-8c84-d2cf859172fa  (ground tee; largest predicted curve (landing lateral offset from the launch direction line))
      balcony: 9bf5e963-e351-42a9-8e26-0afc9f4624fa  (elevated tee T3, ball speed 55 to 65 m/s; median predicted carry)
             tee  ball speed (m/s)  spin (rpm)  carry (m)  ground carry (m)  impact speed (m/s)  impact angle from vertical (deg)  first bounce (m)  bounce (m)  roll (m)  total (m)
shot                                                                                                                   

In [4]:
variants = {
    "restitution x0.8": BounceParams(restitution_scale=0.8),
    "restitution x1.2": BounceParams(restitution_scale=1.2),
    "turf angle x0.5": BounceParams(theta_scale=0.5),
    "turf angle x1.5": BounceParams(theta_scale=1.5),
    "friction 0.25": BounceParams(friction=0.25),
    "rolling retention 0.6": BounceParams(rolling_retention=0.6),
    "no backspin": BounceParams(use_spin=False),
    "rho_g 0.065 (fast green)": BounceParams(rolling_friction=0.065),
    "rho_g 0.196 (slow green)": BounceParams(rolling_friction=0.196),
    "stop height 1 mm": BounceParams(stop_height=0.001),
    "stop height 10 mm": BounceParams(stop_height=0.010),
}
sensitivity = {}
for slot in ["wedge", "mid iron", "driver"]:
    tj = by_id[gallery[slot]]
    base = with_bounce(tj, PENNER)["total"]
    sensitivity[slot] = {"baseline total (m)": base,
                         **{name: with_bounce(tj, params)["total"] - base for name, params in variants.items()}}
sensitivity = pd.DataFrame(sensitivity)
print("change in total distance (m) against Penner's values:")
print(sensitivity.round(1).to_string())

change in total distance (m) against Penner's values:
                          wedge  mid iron  driver
baseline total (m)         93.4     191.2   255.6
restitution x0.8            0.0      -0.5     1.0
restitution x1.2           -0.1       0.7    -1.0
turf angle x0.5             5.7      43.2   118.7
turf angle x1.5            -2.9     -19.6   -31.0
friction 0.25               2.4       0.2     0.0
rolling retention 0.6      -0.9     -15.3   -15.5
no backspin                 7.0      -8.9   -14.5
rho_g 0.065 (fast green)    0.1       2.1     2.1
rho_g 0.196 (slow green)   -0.0      -0.7    -0.7
stop height 1 mm            0.0       0.9     0.9
stop height 10 mm           0.0      -1.2    -1.2


**Reading.** See the tables above.

* **Carry dominates** the total distance: the wedge runs almost nothing, while the mid iron and driver run about 20 m.
* **The turf-compliance angle dominates the sensitivity.** Halving it adds tens of metres (over 100 m for the driver), because a smaller angle keeps more horizontal speed through each impact. Penner derived this rule from a single measured impact (2002b, p. 934), so it is the weakest link.
* **It also shapes the run.** The same rotation keeps shallow hops going, so shots make many tiny hops and reach the 5 mm stop height at a similar low speed. The final roll is therefore about 2 m for almost every shot, and most of the run is bouncing.
* **Other parameters.** Backspin and the rolling retention factor matter at the 10 m level, while restitution, rolling friction and stop height matter little.
* **Status.** None of these values are calibrated to this range, so bounce and roll are illustrative, not predictions.

## 3. Animations

In [5]:
sizes = {}
for slot, track in gallery.items():
    row = test[test["track_id"] == track].iloc[0]
    tj = by_id[track]
    tee = nearest_tee(row["launch_x"], row["launch_y"])
    title = (f"{slot} (test shot {track[:8]}, tee {tee}): carry {tj.distances['carry']:.0f} m, "
             f"total {tj.distances['total']:.0f} m")
    path = ANIM_DIR / f"gallery_{slot.replace(' ', '_')}.html"
    sizes[path.name] = write_html(plotly_figure(tj, tee, title=title, start_at_end=True), path)

# A training shot, with the true apex and landing available through the toggle.
example = train.iloc[0]
example_frame = add_shot_frame(train.iloc[[0]]).iloc[0]
truth = {"apex": tuple(example_frame[["apex_d", "apex_l", "apex_h"]]),
         "landing": tuple(example_frame[["landing_d", "landing_l", "landing_h"]])}
tj = by_id[example["track_id"]]
tee = nearest_tee(example["launch_x"], example["launch_y"])
path = ANIM_DIR / "train_example_with_truth.html"
sizes[path.name] = write_html(plotly_figure(tj, tee, truth=truth, start_at_end=True,
                                            title=f"training shot {example['track_id'][:8]} (tee {tee}), truth toggle top right"), path)
print(pd.Series(sizes, name="bytes").apply(lambda b: f"{b / 1e6:.2f} MB").to_string())

gallery_wedge.html               0.71 MB
gallery_mid_iron.html            2.82 MB
gallery_driver.html              3.12 MB
gallery_strong_curve.html        3.41 MB
gallery_balcony.html             3.06 MB
train_example_with_truth.html    2.41 MB


In [6]:
gif_slot = "mid iron"
tj = by_id[gallery[gif_slot]]
row = test[test["track_id"] == gallery[gif_slot]].iloc[0]
gif_path = ANIM_DIR / "trajectory_mid_iron.gif"
size = write_gif(tj, nearest_tee(row["launch_x"], row["launch_y"]), gif_path, fps=10,
                 title=f"Mid iron, test shot {gallery[gif_slot][:8]}: measured to the net, predicted beyond")
print(f"{gif_path.name}: {size / 1e6:.2f} MB")

trajectory_mid_iron.gif: 0.48 MB


In [7]:
colors = {"wedge": "#2a78d6", "mid iron": "#eb6834", "driver": "#1baf7a", "strong curve": "#eda100", "balcony": "#e87ba4"}
fig, (ax_side, ax_top) = plt.subplots(2, 1, figsize=(10, 6.2), sharex=True, height_ratios=[1, 1.1])
for ax in (ax_side, ax_top):
    ax.axvspan(-5, 60, color="#2a78d6", alpha=0.05, lw=0)
    ax.axvline(60, color="#0b0b0b", lw=1.2)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", lw=0.8)
for slot, track in gallery.items():
    row = test[test["track_id"] == track].iloc[0]
    scene = ScenePath.from_trajectory(by_id[track], nearest_tee(row["launch_x"], row["launch_y"]))
    p = scene.path
    for phases, style, width in [(["radar"], "-", 2.4), (["flight"], "--", 1.8), (["bounce", "roll"], ":", 1.8)]:
        part = p[p["phase"].isin(phases)]
        ax_side.plot(part["D"], part["Z"], style, color=colors[slot], lw=width)
        ax_top.plot(part["D"], part["L"], style, color=colors[slot], lw=width)
    rest = p.iloc[-1]
    ax_top.annotate(slot, (rest["D"], rest["L"]), xytext=(4, 0), textcoords="offset points", va="center",
                    fontsize=9, color="#0b0b0b")
ax_side.annotate("net", (60, ax_side.get_ylim()[1] * 0.92), xytext=(4, 0), textcoords="offset points", fontsize=9)
ax_side.plot([], [], "-", color="#52514e", label="radar, measured")
ax_side.plot([], [], "--", color="#52514e", label="predicted flight")
ax_side.plot([], [], ":", color="#52514e", label="bounce and roll")
ax_side.legend(frameon=False, fontsize=9, loc="upper right")
ax_side.set_ylabel("height above ground (m)")
ax_top.set_ylabel("lateral (m, + left)")
ax_top.set_xlabel("downrange from the tee line (m)")
ax_side.set_title("Five test shots drawn from their inputs only", fontweight="bold", loc="left")
fig.tight_layout()
fig.savefig(FIG_DIR / "fig16_trajectory_gallery.png", dpi=150, bbox_inches="tight", facecolor="#fcfcfb")
plt.show()

/tmp/ipykernel_13902/1145867038.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

* `inrange.trajectory.shot_trajectory(row)` turns one input row into a timed path with phases (radar, flight, bounce, roll). The path meets the measured checkpoints and the submitted apex and landing exactly.
* Bounce and roll follow Penner (2002b). This is not validated here, because the data has no bounce information.
* `python animate_shot.py --track-id <uuid>` writes an interactive 3D animation for any shot. The gallery in `outputs/animations/` covers a wedge, a mid iron, a driver, a strong curve and a balcony shot, and `trajectory_mid_iron.gif` is the writeup animation.